In [131]:
import numpy, gsw

In [143]:
# ad-hoc implementation
def AOU_estimate(potential_temperature, salinity, density, oxygen, mode):
    # function to compute AOU in umol/kg
    # based on Sarmiento & Gruber (Garcia and Gordon, 1992)
    # all inputs can be either scalar or 1D iterables of the same length
    # potential_temperature in degrees Celsius
    # salinity in situ in PSU
    # density in situ in kg/m3
    # oxygen in situ in umol/kg

    # convert to arrays; None -> nan
    pt = numpy.asarray(potential_temperature, dtype=float)
    sa = numpy.asarray(salinity, dtype=float)
    ro = numpy.asarray(density, dtype=float)
    o2 = numpy.asarray(oxygen, dtype=float)

    scalar = (pt.ndim == sa.ndim == ro.ndim == o2.ndim == 0)

    # force 1D so we only write the algebra once
    pt = numpy.atleast_1d(pt)
    sa = numpy.atleast_1d(sa)
    ro = numpy.atleast_1d(ro)
    o2 = numpy.atleast_1d(o2)

    # enforce equal shapes (no broadcasting surprises)
    if not (pt.shape == sa.shape == ro.shape == o2.shape):
        raise ValueError(f"Inputs must have the same shape; got {pt.shape}, {sa.shape}, {ro.shape}, {o2.shape}")

    # elementwise validity mask
    valid = numpy.isfinite(pt) & numpy.isfinite(sa) & numpy.isfinite(ro) & numpy.isfinite(o2)

    out = numpy.full(pt.shape, numpy.nan, dtype=float)

    if mode==1:
        # constants.  <-- KM: cm3/dm3 constants
        A0 = 2.00907
        A1 = 3.22014
        A2 = 4.05010
        A3 = 4.94457
        A4 = -0.256847
        A5 = 3.88767
        B0 = -6.24523e-3
        B1 = -7.37614e-3
        B2 = -1.03410e-2
        B3 = -8.17083e-3
        C0 = -4.88682e-7
    else:
        A0 = 5.80871
        A1 = 3.20291
        A2 = 4.17887
        A3 = 5.10006
        A4 = -9.86643e-2
        A5 = 3.80369
        B0 = -7.01577e-3
        B1 = -7.70028e-3
        B2 = -1.13864e-2
        B3 = -9.51519e-3
        C0 = -2.75915e-7
        
        pt = 1.00024*pt

    # compute only on valid positions
    ptv, sav, rov, o2v = pt[valid], sa[valid], ro[valid], o2[valid]

    Ts = numpy.log((298.15 - ptv) / (273.15 + ptv))
    l  = (A0
          + A1*Ts + A2*(Ts**2) + A3*(Ts**3) + A4*(Ts**4) + A5*(Ts**5)
          + sav*(B0 + B1*Ts + B2*(Ts**2) + B3*(Ts**3))
          + C0*(sav**2))

    scale = 1
    if mode==1:
        scale = (1000/22.3916) # <-- KM: here we're converting using the molar volume of O2, but this I guess has some assumptions in it - STP?
    O2_eq_mmol_per_m3 = scale * numpy.exp(l)    
    O2_eq_umol_per_kg = O2_eq_mmol_per_m3 * rov / 1000
    out[valid] = O2_eq_umol_per_kg - o2v

    if scalar:
        return float(out[0])
    return out

In [144]:
# gsw implementation
def AOU_estimate_gsw(SA, CT, p, lon, lat, oxygen):

    o2sol = gsw.O2sol(SA, CT, p, lon, lat)
    O2_eq_umol_per_kg = o2sol * gsw.rho(SA, CT, p) / 1000

    return O2_eq_umol_per_kg - oxygen

In [148]:
# example based off https://www.teos-10.org/pubs/gsw/html/gsw_O2sol.html:

SA = [34.7118, 34.8915, 35.0256, 34.8472, 34.7366, 34.7324]
CT = [28.8099, 28.4392, 22.7862, 10.2262, 6.8272, 4.3236]
p =  [10, 50, 125, 250, 600, 1000]
lat =  [4, 4, 4, 4, 4, 4]
long = [188, 188, 188, 188, 188, 188]
potential_temperature = gsw.pt_from_CT(SA, CT)
salinity = gsw.SP_from_SA(SA, p, long, lat)
density = gsw.rho(SA, CT, p)
oxygen = [0,0,0,0,0,0]

adhoc_AOU = AOU_estimate(potential_temperature, salinity, density, oxygen, mode=1)
gsw_AOU = AOU_estimate_gsw(SA, CT, p, long, lat, oxygen)

In [149]:
print(adhoc_AOU)

[203.5480463  204.63112552 225.38343405 288.98450546 312.59056458
 332.27268772]


In [150]:
print(gsw_AOU)

[199.22315737 200.23354249 220.141552   281.47631455 304.32843056
 323.38669933]


In [137]:
# example from an argo profile
%pip install -i https://test.pypi.org/simple/ argovisHelpers==0.0.34a13

Looking in indexes: https://test.pypi.org/simple/
Note: you may need to restart the kernel to use updated packages.


In [138]:
from argovisHelpers import helpers as avh

In [139]:
qsp = {
    'id':'4902625_066',
    'data': 'all'
}
p = avh.Profile(avh.query('/argo',qsp)[0])

In [140]:
SA = gsw.SA_from_SP(p.getvar('salinity_sfile'), p.getvar('pressure'), p.longitude, p.latitude)
CT = gsw.CT_from_t(SA, p.getvar('temperature_sfile'), p.getvar('pressure'))
pressure =  p.getvar('pressure')
lat =  numpy.array([p.latitude for i in range(len(SA))])
long = numpy.array([p.longitude for i in range(len(SA))])
potential_temperature = gsw.pt_from_CT(SA, CT)
salinity = gsw.SP_from_SA(SA, pressure, long, lat)
density = gsw.rho(SA, CT, pressure)
oxygen = p.getvar('doxy')

mask = [x is not None for x in oxygen]

In [141]:
adhoc_AOU = AOU_estimate(potential_temperature[mask], salinity[mask], density[mask], oxygen[mask], mode=0)
gsw_AOU = AOU_estimate_gsw(SA[mask], CT[mask], pressure[mask], long[mask], lat[mask], oxygen[mask])

In [129]:
adhoc_AOU

array([ 17.3477633 ,  17.36997975,  16.52326463,  16.51521053,
        16.58200524,  16.64195165,  16.871012  ,  16.90741408,
        16.90061773,  16.99501725,  16.90678179,  17.06061283,
        17.07265665,  17.02746508,  17.1583376 ,  17.18204343,
        17.23939097,  17.31490408,  17.29047452,  17.43302888,
        17.4900972 ,  17.45342542,  17.5685403 ,  17.6476829 ,
        17.6864604 ,  17.7168133 ,  17.80134088,  17.91201306,
        17.73878432,  17.80130628,  17.72523423,  17.62751342,
        17.57720754,  17.26275467,  17.35602702,  17.12079906,
        16.87550984,  16.65079204,  16.64539806,  16.57373786,
        16.60404817,  16.27211725,  16.58159021,  16.91027267,
        16.89837129,  17.08053208,  17.26313954,  17.45616552,
        17.62584178,  17.9875963 ,  18.28317287,  18.82518054,
        19.05140256,  19.39454118,  19.69239111,  20.00625424,
        20.26277132,  20.53552922,  20.7332251 ,  20.81736448,
        20.99371793,  21.01264787,  21.20085705,  21.40

In [130]:
gsw_AOU

array([17.347763298857558, 17.36997974551224, 16.523264633212307,
       16.515210529414475, 16.582005239229403, 16.641951649074855,
       16.87101199825122, 16.90741408197607, 16.90061773260925,
       16.99501724704001, 16.90678179220444, 17.060612832944372,
       17.072656652271945, 17.027465075203423, 17.15833760055844,
       17.182043427747658, 17.23939096595859, 17.314904084379066,
       17.29047451971971, 17.43302887524399, 17.49009719764257,
       17.45342541719606, 17.568540300681804, 17.647682904371493,
       17.686460400072917, 17.716813303521207, 17.801340877489537,
       17.912013057111807, 17.738784319220997, 17.801306277999373,
       17.72523422795595, 17.6275134207379, 17.577207536225984,
       17.262754673406903, 17.356027024134278, 17.12079905585381,
       16.875509843794163, 16.650792042992236, 16.645398063707802,
       16.573737859196967, 16.604048172459073, 16.27211725082762,
       16.5815902076001, 16.910272666506728, 16.898371289223917,
       17.0805

In [142]:
adhoc_AOU - gsw_AOU

array([2.2737367544323206e-13, 4.547473508864641e-13,
       2.2737367544323206e-13, -2.2737367544323206e-13,
       -2.2737367544323206e-13, 0.0, 2.8421709430404007e-13,
       -2.2737367544323206e-13, 0.0, -3.410605131648481e-13,
       -2.2737367544323206e-13, -2.2737367544323206e-13, 0.0,
       -2.2737367544323206e-13, 2.2737367544323206e-13, 0.0, 0.0, 0.0,
       0.0, 2.2737367544323206e-13, 2.2737367544323206e-13,
       -2.2737367544323206e-13, -4.547473508864641e-13, 0.0,
       2.2737367544323206e-13, 0.0, -4.547473508864641e-13, 0.0,
       2.2737367544323206e-13, 0.0, 0.0, 2.2737367544323206e-13, 0.0,
       2.2737367544323206e-13, 2.2737367544323206e-13, 0.0,
       2.2737367544323206e-13, 0.0, 0.0, 0.0, -2.2737367544323206e-13,
       0.0, 0.0, 0.0, 0.0, 4.547473508864641e-13, 3.410605131648481e-13,
       2.2737367544323206e-13, 0.0, -2.8421709430404007e-13, 0.0,
       -2.2737367544323206e-13, 0.0, 0.0, 0.0, 2.2737367544323206e-13,
       -2.2737367544323206e-13, -2.273